# Create tidy "joblib" files to replace the pickle files which are acting weird
## `3_create..` is a copy of the other version, except we need to fix the files since they are not working with new versions of pandas

created by Cassie Lumbrazo\
last updated: July 2025\
run location: UAS linux\
python environment: **cer_treatment**

In [1]:
# import packages 
# %matplotlib widget
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import csv 
import copy 
import os.path 

from mpl_toolkits.axes_grid1 import make_axes_locatable

import pickle

Open CSVs using code from John's Github, starting here

In [2]:
pwd

'/home/cassie/python/repos/CER_timeseries_analysis'

In [4]:
#import csv file for Site Openness and Snow Disappearance
# SDD23 = pd.read_csv("E:\\CassieLumbrazo\\Data\\field_data_proccessed_byJohn\\FinalData\\GapFraction_SDD.csv")
# SDD23 = pd.read_csv("E:\\CassieLumbrazo\\other\\Data\\field_data_proccessed_byJohn\\FinalData\\GapFraction_SDD.csv")
SDD23 = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/GapFraction_SDD_copy_edit.csv")

# I went in to fill the missing gap fraction from John's code with Susan's (from her data) for the forest and gap sites 
# These changes are in the "edit" file 
# SDD23 = pd.read_csv("E:\\CassieLumbrazo\\other\\Data\\field_data_proccessed_byJohn\\FinalData\\GapFraction_SDD_edit.csv")

#make sure all dates are type datetime
SDD23['SDD1'] = pd.to_datetime(SDD23['SDD1'])
SDD23['SDD2'] = pd.to_datetime(SDD23['SDD2'])
SDD23['SDD3'] = pd.to_datetime(SDD23['SDD3'])
SDD23['SDD4'] = pd.to_datetime(SDD23['SDD4'])
SDD23['SDD5'] = pd.to_datetime(SDD23['SDD5'])
SDD23['SDD6'] = pd.to_datetime(SDD23['SDD6'])
SDD23['SDD7'] = pd.to_datetime(SDD23['SDD7'])
SDD23['SDD8'] = pd.to_datetime(SDD23['SDD8'])
SDD23['SDD9'] = pd.to_datetime(SDD23['SDD9'])

#create variables containing SDD23 for north and south respectively
SDD_CN23 = SDD23.loc[SDD23['Site'] == "CN"]
SDD_CS23 = SDD23.loc[SDD23['Site'] == "CS"]

Now, snow depth csvs

In [5]:
# the sites that match susan's sites
CNF23 = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CN-FProcessedData.csv")
CNG23 = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CN-GProcessedData.csv")

CSF23  = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CS-FProcessedData.csv")
CSG23  = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CS-GProcessedData.csv")

# the other sites with forest perscriptions
CN20  = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CN-20ProcessedData.csv")
CN50  = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CN-50ProcessedData.csv")
CN60  = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CN-60ProcessedData.csv")
CN70  = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CN-70ProcessedData.csv")

CS20   = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CS-20ProcessedData.csv")
CS50   = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CS-50ProcessedData.csv")
CS60   = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CS-60ProcessedData.csv")
CS120  = pd.read_csv("/home/cassie/data/cer_treatment_manuscript/csv/CS-120ProcessedData.csv")

In [6]:
# first, make date column datetime object
CNG23['Date'] = pd.to_datetime(CNG23['Date'])
CNF23['Date'] = pd.to_datetime(CNF23['Date'])
CSG23['Date'] = pd.to_datetime(CSG23['Date'])
CSF23['Date'] = pd.to_datetime(CSF23['Date'])

CN20['Date'] = pd.to_datetime(CN20['Date'])
CN50['Date'] = pd.to_datetime(CN50['Date'])
CN60['Date'] = pd.to_datetime(CN60['Date'])
CN70['Date'] = pd.to_datetime(CN70['Date'])

CS20['Date'] = pd.to_datetime(CS20['Date'])
CS50['Date'] = pd.to_datetime(CS50['Date'])
CS60['Date'] = pd.to_datetime(CS60['Date'])
CS120['Date'] = pd.to_datetime(CS120['Date'])

# set date column as index
CNF23.index = pd.DatetimeIndex(CNF23['Date'])
CNG23.index = pd.DatetimeIndex(CNG23['Date'])
CSF23.index = pd.DatetimeIndex(CSF23['Date'])
CSG23.index = pd.DatetimeIndex(CSG23['Date'])

CN20.index = pd.DatetimeIndex(CN20['Date'])
CN50.index = pd.DatetimeIndex(CN50['Date'])
CN60.index = pd.DatetimeIndex(CN60['Date'])
CN70.index = pd.DatetimeIndex(CN70['Date'])

CS20.index = pd.DatetimeIndex(CS20['Date'])
CS50.index = pd.DatetimeIndex(CS50['Date'])
CS60.index = pd.DatetimeIndex(CS60['Date'])
CS120.index = pd.DatetimeIndex(CS120['Date'])


# Not doing this... 
# # drop extra date column
# CNF23.drop(columns=['Date'], inplace=True)
# CNG23.drop(columns=['Date'], inplace=True)
# CSF23.drop(columns=['Date'], inplace=True)
# CSG23.drop(columns=['Date'], inplace=True)

In [7]:
#Calculate statistics for snow depth (minimum depth, maximum depth, and median depth)
#Because there are only three poles for each site, each statistic represents the value of a single pole
# north sites
CNF23['Median_depth'] = CNF23[["Pole1","Pole2","Pole3"]].median(axis=1)
CNF23['Maximum_depth'] = CNF23[["Pole1","Pole2","Pole3"]].max(axis=1)
CNF23['Minimum_depth'] = CNF23[["Pole1","Pole2","Pole3"]].min(axis=1)
CNG23['Median_depth'] = CNG23[["Pole1","Pole2","Pole3"]].median(axis=1)
CNG23['Maximum_depth'] = CNG23[["Pole1","Pole2","Pole3"]].max(axis=1)
CNG23['Minimum_depth'] = CNG23[["Pole1","Pole2","Pole3"]].min(axis=1)

CN20['Median_depth'] = CN20[["Pole1","Pole2","Pole3"]].median(axis=1)
CN20['Maximum_depth'] = CN20[["Pole1","Pole2","Pole3"]].max(axis=1)
CN20['Minimum_depth'] = CN20[["Pole1","Pole2","Pole3"]].min(axis=1)
CN50['Median_depth'] = CN50[["Pole1","Pole2","Pole3"]].median(axis=1)
CN50['Maximum_depth'] = CN50[["Pole1","Pole2","Pole3"]].max(axis=1)
CN50['Minimum_depth'] = CN50[["Pole1","Pole2","Pole3"]].min(axis=1)
CN60['Median_depth'] = CN60[["Pole1","Pole2","Pole3"]].median(axis=1)
CN60['Maximum_depth'] = CN60[["Pole1","Pole2","Pole3"]].max(axis=1)
CN60['Minimum_depth'] = CN60[["Pole1","Pole2","Pole3"]].min(axis=1)
CN70['Median_depth'] = CN70[["Pole1","Pole2","Pole3"]].median(axis=1)
CN70['Maximum_depth'] = CN70[["Pole1","Pole2","Pole3"]].max(axis=1)
CN70['Minimum_depth'] = CN70[["Pole1","Pole2","Pole3"]].min(axis=1)

# south sites
CSF23['Median_depth'] = CSF23[["Pole1","Pole2","Pole3"]].median(axis=1)
CSF23['Maximum_depth'] = CSF23[["Pole1","Pole2","Pole3"]].max(axis=1)
CSF23['Minimum_depth'] = CSF23[["Pole1","Pole2","Pole3"]].min(axis=1)
CSG23['Median_depth'] = CSG23[["Pole1","Pole2","Pole3"]].median(axis=1)
CSG23['Maximum_depth'] = CSG23[["Pole1","Pole2","Pole3"]].max(axis=1)
CSG23['Minimum_depth'] = CSG23[["Pole1","Pole2","Pole3"]].min(axis=1)
CS20['Median_depth'] = CS20[["Pole1","Pole2","Pole3"]].median(axis=1)

CS20['Maximum_depth'] = CS20[["Pole1","Pole2","Pole3"]].max(axis=1)
CS20['Minimum_depth'] = CS20[["Pole1","Pole2","Pole3"]].min(axis=1)
CS50['Median_depth'] = CS50[["Pole1","Pole2","Pole3"]].median(axis=1)
CS50['Maximum_depth'] = CS50[["Pole1","Pole2","Pole3"]].max(axis=1)
CS50['Minimum_depth'] = CS50[["Pole1","Pole2","Pole3"]].min(axis=1)
CS60['Median_depth'] = CS60[["Pole1","Pole2","Pole3"]].median(axis=1)
CS60['Maximum_depth'] = CS60[["Pole1","Pole2","Pole3"]].max(axis=1)
CS60['Minimum_depth'] = CS60[["Pole1","Pole2","Pole3"]].min(axis=1)
CS120['Median_depth'] = CS120[["Pole1","Pole2","Pole3"]].median(axis=1)
CS120['Maximum_depth'] = CS120[["Pole1","Pole2","Pole3"]].max(axis=1)
CS120['Minimum_depth'] = CS120[["Pole1","Pole2","Pole3"]].min(axis=1)

And, Susan's data from 2021 

In [8]:
# # read in data from WY21
# CNF21 = pd.read_csv("E:\\CassieLumbrazo\\Data\\field_data_Dickersonetal2023\\SnowDepthFromTimelapsePhotos\\CNF_WDNR-M6_WY21.csv", skiprows=[0])
# CNG21 = pd.read_csv("E:\\CassieLumbrazo\\Data\\field_data_Dickersonetal2023\\SnowDepthFromTimelapsePhotos\\CNG_WDNR-M1_WY21.csv", skiprows=[0])

# CSF21 = pd.read_csv("E:\\CassieLumbrazo\\Data\\field_data_Dickersonetal2023\\SnowDepthFromTimelapsePhotos\\CSF_WDNR-A4_WY21.csv", skiprows=[0])
# CSG21 = pd.read_csv("E:\\CassieLumbrazo\\Data\\field_data_Dickersonetal2023\\SnowDepthFromTimelapsePhotos\\CSG_WDNR-M4_WY21.csv", skiprows=[0])

# # make date column datetime object
# CNF21['Date'] = pd.to_datetime(CNF21['Date'])
# CNG21['Date'] = pd.to_datetime(CNG21['Date'])
# CSF21['Date'] = pd.to_datetime(CSF21['Date'])
# CSG21['Date'] = pd.to_datetime(CSG21['Date'])

# # set date column as index
# CNF21.index = pd.DatetimeIndex(CNF21['Date'])
# CNG21.index = pd.DatetimeIndex(CNG21['Date'])
# CSF21.index = pd.DatetimeIndex(CSF21['Date'])
# CSG21.index = pd.DatetimeIndex(CSG21['Date'])

# # # drop extra date column
# # CNF21.drop(columns=['Date'], inplace=True)
# # CNG21.drop(columns=['Date'], inplace=True)
# # CSF21.drop(columns=['Date'], inplace=True)
# # CSG21.drop(columns=['Date'], inplace=True)

In [9]:
# # WY2021
# CNF21['Median_depth']  = CNF21[["PoleL","PoleC","PoleR"]].median(axis=1)
# CNF21['Maximum_depth'] = CNF21[["PoleL","PoleC","PoleR"]].max(axis=1)
# CNF21['Minimum_depth'] = CNF21[["PoleL","PoleC","PoleR"]].min(axis=1)

# # there is no center pole for this site 
# CNG21['Median_depth']  = CNG21[["PoleL","PoleR"]].median(axis=1) 
# CNG21['Maximum_depth'] = CNG21[["PoleL","PoleR"]].max(axis=1)
# CNG21['Minimum_depth'] = CNG21[["PoleL","PoleR"]].min(axis=1)

# CSF21['Median_depth']  = CSF21[["PoleL","PoleC","PoleR"]].median(axis=1)
# CSF21['Maximum_depth'] = CSF21[["PoleL","PoleC","PoleR"]].max(axis=1)
# CSF21['Minimum_depth'] = CSF21[["PoleL","PoleC","PoleR"]].min(axis=1)
# CSG21['Median_depth']  = CSG21[["PoleL","PoleC","PoleR"]].median(axis=1)
# CSG21['Maximum_depth'] = CSG21[["PoleL","PoleC","PoleR"]].max(axis=1)
# CSG21['Minimum_depth'] = CSG21[["PoleL","PoleC","PoleR"]].min(axis=1)

### Write the cleaned dataset to pickle for later, or to new csvs

Now, instead of writing pickle files since they are sensitive, let's write `joblib` files instead.

In [11]:
import joblib

In [13]:
# forest and gap sites 
# joblib.dump(CNF21,"/home/cassie/data/cer_treatment_manuscript/joblib/CNF21.joblib")
# joblib.dump(CNG21,"/home/cassie/data/cer_treatment_manuscript/joblib/CNG21.joblib")
# joblib.dump(CSF21,"/home/cassie/data/cer_treatment_manuscript/joblib/CSF21.joblib")
# joblib.dump(CSG21,"/home/cassie/data/cer_treatment_manuscript/joblib/CSG21.joblib")

joblib.dump(CNF23,"/home/cassie/data/cer_treatment_manuscript/joblib/CNF23.joblib")
joblib.dump(CNG23,"/home/cassie/data/cer_treatment_manuscript/joblib/CNG23.joblib")
joblib.dump(CSF23,"/home/cassie/data/cer_treatment_manuscript/joblib/CSF23.joblib")
joblib.dump(CSG23,"/home/cassie/data/cer_treatment_manuscript/joblib/CSG23.joblib")

# the other post treatment sites 
joblib.dump(CN20,"/home/cassie/data/cer_treatment_manuscript/joblib/CN20.joblib")
joblib.dump(CN50,"/home/cassie/data/cer_treatment_manuscript/joblib/CN50.joblib")
joblib.dump(CN60,"/home/cassie/data/cer_treatment_manuscript/joblib/CN60.joblib")
joblib.dump(CN70,"/home/cassie/data/cer_treatment_manuscript/joblib/CN70.joblib")

joblib.dump(CS20,"/home/cassie/data/cer_treatment_manuscript/joblib/CS20.joblib")
joblib.dump(CS50,"/home/cassie/data/cer_treatment_manuscript/joblib/CS50.joblib")
joblib.dump(CS60,"/home/cassie/data/cer_treatment_manuscript/joblib/CS60.joblib")
joblib.dump(CS120,"/home/cassie/data/cer_treatment_manuscript/joblib/CS120.joblib")

joblib.dump(SDD_CN23,"/home/cassie/data/cer_treatment_manuscript/joblib/SDD_CN23.joblib")
joblib.dump(SDD_CS23,"/home/cassie/data/cer_treatment_manuscript/joblib/SDD_CS23.joblib")

['/home/cassie/data/cer_treatment_manuscript/joblib/SDD_CS23.joblib']

In [ ]:
# # forest and gap sites 
# pickle.dump(CNF21, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CNF21.pkl", "wb"))
# pickle.dump(CNG21, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CNG21.pkl", "wb"))
# pickle.dump(CSF21, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CSF21.pkl", "wb"))
# pickle.dump(CSG21, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CSG21.pkl", "wb"))

# pickle.dump(CNF23, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CNF23.pkl", "wb"))
# pickle.dump(CNG23, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CNG23.pkl", "wb"))
# pickle.dump(CSF23, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CSF23.pkl", "wb"))
# pickle.dump(CSG23, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CSG23.pkl", "wb"))

# # the other post treatment sites 
# pickle.dump(CN20, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CN20.pkl", "wb"))
# pickle.dump(CN50, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CN50.pkl", "wb"))
# pickle.dump(CN60, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CN60.pkl", "wb"))
# pickle.dump(CN70, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CN70.pkl", "wb"))

# pickle.dump(CS20, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CS20.pkl", "wb"))
# pickle.dump(CS50, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CS50.pkl", "wb"))
# pickle.dump(CS60, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CS60.pkl", "wb"))
# pickle.dump(CS120, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\CS120.pkl", "wb"))

In [ ]:
# # now the SDD files too
# pickle.dump(SDD_CN23, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\SDD_CN23.pkl", "wb"))
# pickle.dump(SDD_CS23, open("C:\\Users\\Lumbr\\OneDrive - UW\\Documents\\Washington\\EasternCascades\\Python\\CER_Timeseries_Analysis\\Pickle\\SDD_CS23.pkl", "wb"))

If we want, I can add CSVs to this instead, but let's start here